In [ ]:
import pandas as pd

# 1. Ingesta: Cargar datos (dataset de ejemplo, con identidades de cliente anonimizadas)
df = pd.read_csv('ventas_ejemplo.csv', sep=';', encoding='utf-8', on_bad_lines='skip')

# 2. Transformación: Arreglar herencia de datos de Tiendanube en compras múltiples
# (Rellena los huecos vacíos hacia abajo cuando un cliente compra varios productos en la misma orden)
columnas_a_rellenar = ['Email', 'Fecha', 'Estado del pago', 'Total']
df[columnas_a_rellenar] = df[columnas_a_rellenar].ffill()

# 3. Limpieza: Aislar ventas reales (dinero en caja) y formatear la línea de tiempo
# Solo nos importa el 'Estado del pago' que diga 'Recibido'
df_ventas = df[df['Estado del pago'] == 'Recibido'].copy()
df_ventas['Fecha'] = pd.to_datetime(df_ventas['Fecha'], format='%d/%m/%Y %H:%M:%S', errors='coerce')

# 4. Agrupación: Colapsar a nivel de orden única
# Si una orden tiene 3 productos, colapsamos a 1 sola fila con el Total para no inflar los ingresos
df_modelo = df_ventas[['Email', 'Fecha', 'Número de orden', 'Total']].drop_duplicates(subset=['Número de orden'])

print(f"✅ Fase 1 completada. Órdenes únicas listas para procesar: {df_modelo.shape[0]}")

✅ Fase 1 completada. Órdenes únicas listas para procesar: 2685


In [ ]:
import datetime as dt

# 1. Definir el "Día Cero" (la fecha de la compra más reciente en la tienda + 1 día)
fecha_maxima = df_modelo['Fecha'].max() + dt.timedelta(days=1)

# 2. Agrupar por Cliente (Email) y calcular sus métricas vitales
rfm = df_modelo.groupby('Email').agg({
    'Fecha': lambda x: (fecha_maxima - x.max()).days,  # Recencia: Días desde su última compra
    'Número de orden': 'count',                        # Frecuencia: Cantidad total de compras
    'Total': 'sum'                                     # Monetización: Dinero total dejado en la marca
}).reset_index()

# 3. Renombrar columnas
rfm.rename(columns={
    'Fecha': 'Recencia_Dias',
    'Número de orden': 'Frecuencia',
    'Total': 'Monetizacion_Total'
}, inplace=True)

print(rfm.head())
print(f"\n✅ Fase 2 completada. Clientes únicos perfilados: {rfm.shape[0]}")

                      Email  Recencia_Dias  Frecuencia  Monetizacion_Total
0  cliente_0001@ejemplo.com              1           1            68165.00
1  cliente_0002@ejemplo.com              1           8          1086498.99
2  cliente_0003@ejemplo.com              1           1            63165.00
3  cliente_0004@ejemplo.com              1           1            62353.00
4  cliente_0005@ejemplo.com              2           1           140857.99

✅ Fase 2 completada. Clientes únicos perfilados: 2050


In [ ]:
!pip install lifetimes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 12.9 MB/s eta 0:00:00


In [ ]:
from lifetimes import BetaGeoFitter
from lifetimes.utils import summary_data_from_transaction_data

# 1. Transformación: Adaptamos los datos al formato estricto que exige el algoritmo
df_predictivo = summary_data_from_transaction_data(
    df_modelo,
    'Email',
    'Fecha',
    monetary_value_col='Total',
    observation_period_end=df_modelo['Fecha'].max()
)

# 2. Entrenamiento: Instanciamos el modelo BG/NBD y le enseñamos el comportamiento histórico
bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(df_predictivo['frequency'], df_predictivo['recency'], df_predictivo['T'])

# 3. Predicción: Le pedimos al modelo que proyecte exactamente 180 días (6 meses) hacia el futuro
t_prediccion = 180
df_predictivo['Compras_Predichas_6m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    t_prediccion,
    df_predictivo['frequency'],
    df_predictivo['recency'],
    df_predictivo['T']
)

# 4. Decisión de Negocio: Ordenamos a los clientes de mayor a menor probabilidad de compra
top_clientes = df_predictivo.sort_values(by='Compras_Predichas_6m', ascending=False)

print("--- Top 5 Clientes Predictivos (Próximos 6 meses) ---")
print(top_clientes[['frequency', 'recency', 'T', 'Compras_Predichas_6m']].head())

--- Top 5 Clientes Predictivos (Próximos 6 meses) ---
                          frequency  recency      T  Compras_Predichas_6m
Email                                                                    
cliente_0002@ejemplo.com        7.0    167.0  167.0               3.174618
cliente_0170@ejemplo.com       13.0    607.0  640.0               2.380183
cliente_0247@ejemplo.com       10.0    205.0  260.0               2.269568
cliente_0162@ejemplo.com        5.0    173.0  204.0               1.730866
cliente_0113@ejemplo.com        3.0     28.0   49.0               1.729720


In [ ]:
from lifetimes import GammaGammaFitter

# 1. Filtro matemático: El algoritmo monetario solo puede aprender de clientes que han comprado más de una vez
clientes_recurrentes = df_predictivo[df_predictivo['frequency'] > 0].copy()

# 2. Entrenamiento Monetario: Instanciamos el modelo Gamma-Gamma
ggf = GammaGammaFitter(penalizer_coef=0.0)
ggf.fit(clientes_recurrentes['frequency'], clientes_recurrentes['monetary_value'])

# 3. Predicción del Ticket: Calculamos cuánto gastará en promedio cada cliente en cada compra futura
clientes_recurrentes['Ticket_Promedio_Predicho'] = ggf.conditional_expected_average_profit(
    clientes_recurrentes['frequency'],
    clientes_recurrentes['monetary_value']
)

# 4. El Santo Grial (LTV a 6 meses): Multiplicamos cantidad de compras futuras por ticket promedio futuro
clientes_recurrentes['LTV_6_Meses_Predicho'] = clientes_recurrentes['Compras_Predichas_6m'] * clientes_recurrentes['Ticket_Promedio_Predicho']

# 5. Decisión de Negocio: Ordenamos el archivo por los clientes más rentables
top_ltv = clientes_recurrentes.sort_values(by='LTV_6_Meses_Predicho', ascending=False)

print("--- Top 5 Clientes Más Rentables (Próximos 6 meses) ---")
print(top_ltv[['frequency', 'monetary_value', 'Compras_Predichas_6m', 'LTV_6_Meses_Predicho']].head())

--- Top 5 Clientes Más Rentables (Próximos 6 meses) ---
                          frequency  monetary_value  Compras_Predichas_6m  LTV_6_Meses_Predicho
Email                                                                                          
cliente_0247@ejemplo.com       10.0   288511.900000               2.269568          636606.735251
cliente_0002@ejemplo.com        7.0   128706.998571               3.174618          409109.129711
cliente_0394@ejemplo.com       15.0   230315.734000               1.706390          387159.953317
cliente_0158@ejemplo.com        6.0   214500.000000               1.555274          322952.851532
cliente_0170@ejemplo.com       13.0   113300.614615               2.380183          271342.767374


In [ ]:
# 1. Redondear matemáticamente las columnas a 2 decimales
columnas_a_redondear = {
    'monetary_value': 2,
    'Compras_Predichas_6m': 2,
    'Ticket_Promedio_Predicho': 2,
    'LTV_6_Meses_Predicho': 2
}
clientes_recurrentes = clientes_recurrentes.round(columnas_a_redondear)

# 2. Actualizar el Top 5 con los datos redondeados
top_ltv = clientes_recurrentes.sort_values(by='LTV_6_Meses_Predicho', ascending=False)

print("--- Top 5 Clientes Más Rentables (Datos Limpios) ---")
print(top_ltv[['frequency', 'monetary_value', 'Compras_Predichas_6m', 'LTV_6_Meses_Predicho']].head())

# 3. Exportar el archivo final
top_ltv.to_csv('LTV_Predicho_DenimWest.csv', index=True, encoding='utf-8')
print("\n✅ Archivo 'LTV_Predicho_DenimWest.csv' exportado con éxito.")

--- Top 5 Clientes Más Rentables (Datos Limpios) ---
                          frequency  monetary_value  Compras_Predichas_6m  LTV_6_Meses_Predicho
Email                                                                                          
cliente_0247@ejemplo.com       10.0       288511.90                   2.27             636606.74
cliente_0002@ejemplo.com        7.0       128707.00                   3.17             409109.13
cliente_0394@ejemplo.com       15.0       230315.73                   1.71             387159.95
cliente_0158@ejemplo.com        6.0       214500.00                   1.56             322952.85
cliente_0170@ejemplo.com       13.0       113300.61                   2.38             271342.77

✅ Archivo 'LTV_Predicho_DenimWest.csv' exportado con éxito.
